# Reconhecimento de entidades nomeadas

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_10/01_reconhecimento_de_entidades.ipynb)

NER localiza e tipifica menções. Regras são transparentes e frágeis; dicionários têm
alta precisão no domínio conhecido e baixa cobertura de novidades; modelos aprendem
padrões, mas exigem dados e avaliação. Aqui combinaremos regex e autoridade local.

![Regras, dicionários e modelos oferecem coberturas distintas.](imagens/01_metodos_ner.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_10'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import re,pandas as pd
docs=pd.read_csv("dados/documentos.csv"); aut=pd.read_csv("dados/autoridades.csv").fillna("")
mapa={}
for _,r in aut.iterrows():
    for forma in [r.nome_canonico,*[x for x in r.variantes.split(";") if x]]: mapa[forma]=(r.id_entidade,r.tipo,r.nome_canonico)
padrao=re.compile("|".join(sorted(map(re.escape,mapa),key=len,reverse=True)))
extracoes=[]
for _,d in docs.iterrows():
    for m in padrao.finditer(d.texto):
        ident,tipo,canon=mapa[m.group()];extracoes.append([d.id_documento,m.group(),m.start(),m.end(),ident,tipo,canon,"dicionário"])
    for m in re.finditer(r"\b\d{1,2} de [a-zç]+ de \d{4}\b",d.texto.lower()): extracoes.append([d.id_documento,d.texto[m.start():m.end()],m.start(),m.end(),"","DATA","","regex"])
entidades=pd.DataFrame(extracoes,columns=["id_documento","mencao","inicio","fim","id_entidade","tipo","nome_canonico","metodo"])
display(entidades.head(12));display(entidades.tipo.value_counts())

A posição inicial/final torna a extração rastreável. O dicionário não encontra nomes
novos; a regex de data não compreende calendários, apenas padrões. Um modelo treinado
poderia ampliar cobertura, mas seus erros continuariam exigindo amostra anotada.

### U10-A02 — Atividade integrada — cobertura e falsos positivos

Leia todos os documentos, marque uma menção perdida e um possível falso positivo,
proponha regra ou entrada de dicionário e explique o novo erro que ela pode criar.
**Dinâmica (25 minutos):** duplas, uma amplia cobertura e outra tenta quebrar a regra.
**Resposta:** escreva aqui.

## Passagem

Localizar “Ana F.” ainda não informa se é Ana Ferreira. O notebook seguinte separa
normalização gráfica de vinculação a um referente.